In [1]:
import ee
import pandas as pd

ee.Initialize(project='satellite-economic-nowcasting')

# Load Nigeria Admin-1 boundaries
states = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM0_NAME', 'Nigeria'))
print(f"Boundaries ready: {states.size().getInfo()} states")

c:\Users\iaayu\satellite-economic-nowcasting\venv\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1

  warnings.warn(warning, category=DeprecationWarning)


Boundaries ready: 37 states


In [2]:
# 1. Query NOAA VIIRS monthly radiance composites for 2023
viirs_2023 = (
    ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
    .filterDate('2023-01-01', '2023-12-31')
    .select('avg_rad')
    .median()
)

# 2. Compute spatial mean nighttime radiance per state
viirs_stats = viirs_2023.reduceRegions(
    collection=states,
    reducer=ee.Reducer.mean(),
    scale=1000  # 1km resolution
)

# 3. Extract to local DataFrame
features = viirs_stats.reduceColumns(
    reducer=ee.Reducer.toList(2),
    selectors=['ADM1_NAME', 'mean']
).getInfo()['list']

df_viirs = pd.DataFrame(features, columns=['state_name', 'viirs_mean_radiance']).sort_values('viirs_mean_radiance', ascending=False)
print("Top 5 brightest states by mean nighttime radiance:")
print(df_viirs.head())

Top 5 brightest states by mean nighttime radiance:
   state_name  viirs_mean_radiance
30      Lagos             4.457076
20      Delta             2.189424
10     Rivers             2.110607
13      Abuja             1.482304
22        Imo             1.178502
